<a href="https://colab.research.google.com/github/Fordfire337/CS-4410-intro-machine-learning/blob/main/JWHW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.datasets import mnist, fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten, MaxPooling2D
from tensorflow.keras.utils import to_categorical

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.19.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [11]:
SEED = 42
tf.keras.utils.set_random_seed(SEED)

EPOCHS = 5
BATCH_SIZE = 64
VALIDATION_SPLIT = 0.1
NUM_CLASSES = 10
IMAGE_SHAPE = (28, 28, 1)


In [12]:
def load_and_prepare(dataset_name):
    if dataset_name == "MNIST":
        (X_train, y_train), (X_test, y_test) = mnist.load_data()
    elif dataset_name == "Fashion-MNIST":
        (X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    X_train = X_train.reshape((X_train.shape[0], 28, 28, 1)).astype("float32") / 255.0
    X_test = X_test.reshape((X_test.shape[0], 28, 28, 1)).astype("float32") / 255.0

    y_train = to_categorical(y_train, NUM_CLASSES)
    y_test = to_categorical(y_test, NUM_CLASSES)

    return (X_train, y_train), (X_test, y_test)


In [13]:
def build_model(add_dense_4096=False, remove_first_dense=False):
    model = Sequential([
        tf.keras.Input(shape=IMAGE_SHAPE),

        Conv2D(filters=64, kernel_size=(3, 3), activation="relu"),
        MaxPooling2D(pool_size=(2, 2)),

        Conv2D(filters=128, kernel_size=(3, 3), activation="relu"),
        MaxPooling2D(pool_size=(2, 2)),

        Flatten()
    ])

    if not remove_first_dense:
        model.add(Dense(units=128, activation="relu"))

    if add_dense_4096:
        model.add(Dense(units=4096, activation="relu"))

    model.add(Dense(units=10, activation="softmax"))

    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

In [14]:
def train_and_evaluate(dataset_name, model_label, add_dense_4096=False, remove_first_dense=False):
    (X_train, y_train), (X_test, y_test) = load_and_prepare(dataset_name)
    model = build_model(add_dense_4096=add_dense_4096, remove_first_dense=remove_first_dense)

    start = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        verbose=1
    )
    train_seconds = time.perf_counter() - start

    eval_start = time.perf_counter()
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
    eval_seconds = time.perf_counter() - eval_start

    best_val_accuracy = max(history.history["val_accuracy"])
    final_val_accuracy = history.history["val_accuracy"][-1]
    final_train_accuracy = history.history["accuracy"][-1]

    total_params = model.count_params()

    return {
        "dataset": dataset_name,
        "model": model_label,
        "parameters": total_params,
        "train_accuracy_last_epoch": final_train_accuracy,
        "val_accuracy_last_epoch": final_val_accuracy,
        "best_val_accuracy": best_val_accuracy,
        "test_accuracy": test_accuracy,
        "test_loss": test_loss,
        "training_time_seconds": train_seconds,
        "evaluation_time_seconds": eval_seconds
    }


In [15]:
results = []

results.append(train_and_evaluate(
    dataset_name="MNIST",
    model_label="Baseline"
))

results.append(train_and_evaluate(
    dataset_name="Fashion-MNIST",
    model_label="Baseline"
))

results_df = pd.DataFrame(results)
results_df


Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9015 - loss: 0.3220 - val_accuracy: 0.9842 - val_loss: 0.0586
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9852 - loss: 0.0456 - val_accuracy: 0.9858 - val_loss: 0.0524
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9910 - loss: 0.0280 - val_accuracy: 0.9830 - val_loss: 0.0641
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9938 - loss: 0.0193 - val_accuracy: 0.9860 - val_loss: 0.0477
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9955 - loss: 0.0144 - val_accuracy: 0.9902 - val_loss: 0.0436
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9831 - loss: 0.0476
Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.7651 - loss: 0.6453 - val_accuracy: 0.8665 - val_loss: 0.3765
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8833 - loss: 0.3200 - val_accuracy: 0.8848 - val_loss: 0.3183
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━

,dataset,model,parameters,train_accuracy_last_epoch,val_accuracy_last_epoch,best_val_accuracy,test_accuracy,test_loss,training_time_seconds,evaluation_time_seconds
0,MNIST,Baseline,485514,0.995426,0.990167,0.990167,0.9872,0.036532,23.351351,1.914219
1,Fashion-MNIST,Baseline,485514,0.932111,0.905833,0.905833,0.9001,0.302679,23.509634,1.519578


In [16]:
results.append(train_and_evaluate(
    dataset_name="MNIST",
    model_label="Baseline + Dense(4096)",
    add_dense_4096=True
))

results.append(train_and_evaluate(
    dataset_name="Fashion-MNIST",
    model_label="Baseline + Dense(4096)",
    add_dense_4096=True
))

results_df = pd.DataFrame(results)
results_df


Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.8965 - loss: 0.3233 - val_accuracy: 0.9808 - val_loss: 0.0599
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9858 - loss: 0.0475 - val_accuracy: 0.9857 - val_loss: 0.0560
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9911 - loss: 0.0295 - val_accuracy: 0.9878 - val_loss: 0.0490
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9920 - loss: 0.0253 - val_accuracy: 0.9872 - val_loss: 0.0579
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9947 - loss: 0.0178 - val_accuracy: 0.9872 - val_loss: 0.0579
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9825 - loss: 0.0652
Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.7460 - loss: 0.6700 - val_accuracy: 0.8638 - val_loss: 0.3716
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8846 - loss: 0.3165 - val_accuracy: 0.8745 - val_loss: 0.3598
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━

,dataset,model,parameters,train_accuracy_last_epoch,val_accuracy_last_epoch,best_val_accuracy,test_accuracy,test_loss,training_time_seconds,evaluation_time_seconds
0,MNIST,Baseline,485514,0.995426,0.990167,0.990167,0.9872,0.036532,23.351351,1.914219
1,Fashion-MNIST,Baseline,485514,0.932111,0.905833,0.905833,0.9001,0.302679,23.509634,1.519578
2,MNIST,Baseline + Dense(4096),1053578,0.994593,0.987167,0.987833,0.9864,0.050669,26.928377,1.499377
3,Fashion-MNIST,Baseline + Dense(4096),1053578,0.932778,0.905000,0.905000,0.9029,0.296019,25.292242,1.517379


In [17]:
summary_df = results_df.copy()

for col in ["train_accuracy_last_epoch", "val_accuracy_last_epoch", "best_val_accuracy", "test_accuracy"]:
    summary_df[col] = summary_df[col].map(lambda x: round(float(x), 4))

for col in ["test_loss", "training_time_seconds", "evaluation_time_seconds"]:
    summary_df[col] = summary_df[col].map(lambda x: round(float(x), 2))

summary_df = summary_df.sort_values(["dataset", "model"]).reset_index(drop=True)
summary_df


,dataset,model,parameters,train_accuracy_last_epoch,val_accuracy_last_epoch,best_val_accuracy,test_accuracy,test_loss,training_time_seconds,evaluation_time_seconds
0,Fashion-MNIST,Baseline,485514,0.9321,0.9058,0.9058,0.9001,0.30,23.51,1.52
1,Fashion-MNIST,Baseline + Dense(4096),1053578,0.9328,0.9050,0.9050,0.9029,0.30,25.29,1.52
2,MNIST,Baseline,485514,0.9954,0.9902,0.9902,0.9872,0.04,23.35,1.91
3,MNIST,Baseline + Dense(4096),1053578,0.9946,0.9872,0.9878,0.9864,0.05,26.93,1.50
